### **Parsing the response**
Extract thought, action and query from the LLM response

In [ ]:
def parse_response(response):
    parse_list = response.split('\n')
    thought = ""
    action = ""
    query = ""

    for line in parse_list:
        if line.startswith("Thought:"):
            thought = line.split(":",1)[1].strip()

        if line.startswith("Action:"):
            action_input = line.split(":",1)[1].strip()

            if action_input.startswith("Finish"):
                action = "Finish"
                query = action_input.split("(")[1].rstrip(")").strip('"')

            elif action_input.startswith("Search"):
                action = "Search"
                query = action_input.split("(")[1].strip(")").strip('"')

    return thought, action, query

### **Main ReAct controller**

In [ ]:
 def run_agent(question, context):
    # Initialize agent state
    memory = []
    trajectory = []
    search_count = 0

    previous_queries = set()
    # ReAct reasoning loop (maximum 4 retrievals)
    while search_count<4:
        response = ask_llm(question, memory)
        thought, action, query = parse_response(response)

        # if query in previous_queries:
        #     return None, trajectory
        # else:
        #     previous_queries.add(query)

        if action == "Search":
            search_count += 1
            observation = Search(query,tf_idf_vectorizer,vectorized_matrix,context)
            memory.append({"action": action, "query": query, "observation": observation})
            trajectory.append({"thought": thought,"action": action,"observation": observation, "query": query})

        elif action == "Finish":
            trajectory.append({"thought": thought,"action": action,"answer": query})
            return query, trajectory
        # Unexpected action returned by the LLM
        else:
            trajectory.append({"thought": thought,"action": "Invalid"})
            return None, trajectory

    return None, trajectory